### Clustering

- I need to do some clustering algos and see which does best
- Select which columns to use


What am I hoping clustering will do?
- Make more "like" groupings of school districts based on certain demographic data. 
- I believe the clusters will still have normal distribution curves, and linear relationships.

What columns are allowed in clustering -- For this one I want to make two selection types, one with ethnic data and one without as some grants or other funding does not like considering that information when benchmarking
- Eco-dis %
- student body size
- average teacher pay
- Ethnic homogeneity with students and staff
- Staff size


What makes up a row?
- a row will be made up of one school with all of its given metrics
- I want to only select schools who's metrics have not drastically changed in the past 3 years. What is why I have the historical data as I believe if a school has hd drastic changes they will be an outlier and throw off the model. for the school years data I will want to use the 2024 numbers as I believe any miss calculations have been adjusted by then (not sure if they even upload corrections but don't feel like looking as of now).
- I will need to normalize all the numbers down to aq 0 to 1 scale so large differences aren't causing issues.

I would also like to run an algo to see how strongly any one variable seems to be in relation to the overall outcomes

Right now, my data probably looks like a few loosely clouds semi clustered and overlapping because nothing is normalized and teacher pay of $40,000 and eco-dis count of 270 out of 3000 students are working together to make the a shape but are working on completely different scales, so I need an approach that with a very loss clustering algo until I clean further.

In [1]:
import pandas as pd
import sklearn


In [2]:
student_df = pd.read_csv(r"Clean Datasets\student_analysis_df.csv", dtype={'YearDistrict ID': str, 'DISTRICT': str, 'Year': str})
staff_df = pd.read_csv(r"Clean Datasets\staff_analysis_df.csv", dtype={'YearDistrict ID': str, 'DISTRICT': str, 'Year': str})


## Exploratory Clustering

- find rates for student and staff data
- Z-Score normalize
- 

- Stretch goal: Principal Components Analysis (PAC) on eco-dis, homeless


In [3]:
# Defining columns to keep
student_col_to_keep = ['YearDistrict ID', 'DISTRICT', 'Year', 'All Students Count', 'Special Ed Count',
                        'Bilingual/ESL Count', 'Career & Technical Education Count',
                        'Gifted & Talented Count', 'EB/EL Count', 'Econ Disadv Count',
                        'Non-Educationally Disadv Count', 'At Risk Count',
                        'American Indian Count', 'Asian Count', 'Pacific Islander Count',
                        'Two or More Races Count', 'African American Count', 'Hispanic Count',
                        'White Count', 'Male Count', 'Female Count',
                        'Section 504 Count', 'Title I Count', 'Homeless Count',
                        'Immigrant Count', 'Migrant Count',
                        'Total Students with Disabilities Count',
                        'Intellectual Disabilities Count', 'Physical Disabilities Count',
                        'Behavioral Disabilities Count', 'District Name', 'County ID',
                        'County Name', 'Region', 'Charter Flag', 'DFLALTED']

staff_col_to_keep = [   'YearDistrict ID', 'Year', 'DISTRICT',       
                        'Teacher Total Full Time Equiv Count',
                        'Support Total Full Time Equiv Count',
                        'School Admin Total Full Time Equiv Count',
                        'Central Admin Total Full Time Equiv Count',
                        'Educ Aide Total Full Time Equiv Count',
                        'Teacher Total Base Salary Total', 'Support Total Base Salary Total',
                        'School Admin Total Base Salary Total',
                        'Teacher Beginning Full Time Equiv Count',
                        'Teacher 1-5 Years Full Time Equiv Count',
                        'Teacher 6-10 Years Full Time Equiv Count',
                        'Teacher 11-20 Years Full Time Equiv Count',
                        'Teacher 21-30 Years Full Time Equiv Count',
                        'Teacher > 30 Years Full Time Equiv Count',
                        'Teacher Beginning Base Salary Total',
                        'Teacher 1-5 Years Base Salary Total',
                        'Teacher 6-10 Years Base Salary Total',
                        'Teacher 11-20 Years Base Salary Total',
                        'Teacher 21-30 Years Base Salary Total',
                        'Teacher > 30 Years Base Salary Total',
                        'Teacher No Degree Full Time Equiv Count',
                        'Teacher BA Degree Full Time Equiv Count',
                        'Teacher MS Degree Full Time Equiv Count',
                        'Teacher PH Degree Full Time Equiv Count',
                        'Teacher American Indian Full Time Equiv Count',
                        'Teacher Pacific Islander Full Time Equiv Count',
                        'Teacher Asian Full Time Equiv Count',
                        'Teacher African American Full Time Equiv Count',
                        'Teacher Hispanic Full Time Equiv Count',
                        'Teacher White Full Time Equiv Count',
                        'Teacher Two or more races Full Time Equiv Count',
                        'Teacher Male Full Time Equiv Count',
                        'Teacher Female Full Time Equiv Count',
                        'Teacher Regular Program Full Time Equiv Count',
                        'Teacher Career & Technical Prgms Full Time Equiv Count',
                        'Teacher Bilingual Program Full Time Equiv Count',
                        'Teacher Compensatory Program Full Time Equiv Count',
                        'Teacher Gifted & Talented Program Full Time Equiv Count',
                        'Teacher Special Education Full Time Equiv Count',
                        'Teacher Other Full Time Equiv Count', 'Teacher Turnover Numerator',
                        'Teacher Turnover Denominator', 'Principal Experience Total',
                        'Principal Tenure Total', 'Assistant Principal Experience Total',
                        'Assistant Principal Tenure Total',
                        'Teacher Incentive Allotment Master Head Count',
                        'Teacher Incentive Allotment Exemplary Head Count',
                        'Teacher Incentive Allotment Recognized Head Count']

In [4]:
# create filtered dataframes
student_df_1 = student_df[student_col_to_keep]
staff_df_1 = staff_df[staff_col_to_keep]

# student_df_1
# staff_df_1

In [5]:
# Select Student Numerators
student_numerators = [ 'Special Ed Count', 'Bilingual/ESL Count',
                        'Career & Technical Education Count', 'Gifted & Talented Count',
                        'EB/EL Count', 'Econ Disadv Count', 'Non-Educationally Disadv Count',
                        'At Risk Count', 'American Indian Count', 'Asian Count',
                        'Pacific Islander Count', 'Two or More Races Count',
                        'African American Count', 'Hispanic Count', 'White Count', 'Male Count',
                        'Female Count', 'Section 504 Count', 'Title I Count',
                        'Homeless Count', 'Immigrant Count', 'Migrant Count',
                        'Total Students with Disabilities Count',
                        'Intellectual Disabilities Count', 'Physical Disabilities Count',
                        'Behavioral Disabilities Count']

### Smoothed Rate Function (Explore shrinkage estimation and  smoothing tequinices)

Explore shrinkage estimation and  smoothing techneics

Beta–Binomial shrinkage estimator

Notes: A discovery happened as I was exploring my data sets post z-score calculations and that was some districts were in the range of (10, 9) but when I just open the data set it looks like most of the numbers are in the range (1, -1). while I did not clearly know why, from what I have learned so far about statistical modeling, I could tell such huge ranges did not make sense so I looked at the affected columns. 

When I started exploring the affected of columns, I noticed that. often the cell that I calculated off of for example number of teachers with 30 plus years of experience. It may have been zero because it was a small district and they had their teachers. Initially with these. As I wanted to keep the data set. I wanted to keep the entire thing whole. But I knew I could not just make the values Nan as that would. affect Downstream statistical values either creating them inaccurately or removing them from the data set and I wanted to try and keep as many rows or values as possible because otherwise I'd have to remove them from multiple years and that would eventually shrink my data set to what I believe will be unusable because I'm only working within a thousand or so data points.

In school districts with small populations, even a single data point can create an extreme outlier. For example, in a district of only five teachers, one teachers constitutes 20% of the total; that would likely create huge statistical anomaly in may calculations down the line. To prevent these small-population "flukes" from distorting our model, we apply a smoothing technique. This approach provides "statistical help" to smaller districts by pulling their rates toward the state average, ensuring that their limited data size doesn't create misleading extremes that skew the overall analysis.
 

In [6]:
# Function: Find rate of num/denom
def find_rate(df, numerator, denominator):
    """
    Creates a rate column for any given value
    df: DataFrame
    Numerator: top number of a fraction (the part)
    Denominator: bottom number of a fraction (the whole)
    """
    # constant = 20
    if type(numerator) == list:
        for col in numerator:
            # mean_rate = df.groupby("Year")[col].transform('mean')
            # df[col + "_Rate"] = (df[col] + constant * mean_rate) / (df[denominator] + constant)
            # df[col + "_Rate"] = df.groupby("Year")[col].transform('mean')
            df[col + "_Rate"] =df[col]/df[denominator]
    else:
        # mean_rate = df.groupby("Year")[numerator].transform('mean')
        # df[numerator + "_Rate"] = (df[numerator] + constant * mean_rate) / (df[denominator] + constant)
        df[numerator+"_Rate"] = df[numerator]/df[denominator]

    
    return df






In [7]:
def rate_smoothing(df, numerator, denominator):
    constant = 20

    if type(numerator) == list:
        for col in numerator:
            rate_col = col+"_Rate"
            col_suffix = col.replace("_Rate", "") + "_Smoothed_Rate"
            # mean_rate = df.groupby("Year")[rate_col].transform('sum')/df.groupby("Year")[rate_col].transform('count')
            mean_rate = (
                df.groupby("Year")[col].transform("sum") /
                df.groupby("Year")[denominator].transform("sum")
            )
                            
            df[col_suffix] = (df[col] + constant * mean_rate) / (df[denominator] + constant)
    else:
        if "_Rate" in numerator:
            col_suffix = col.replace("_Rate", "") + "_Smoothed_Rate"
            mean_rate = (
                df.groupby("Year")[col].transform("sum") /
                df.groupby("Year")[denominator].transform("sum")
            )
            df[col_suffix] = (df[numerator] + constant * mean_rate) / (df[denominator] + constant)
    
    
    return df

### Beta Binomial Model

This has taken a minute to learn.

Fake successes = p×m= 0.42×100 = 42
Fake failures = (1−p)×m=58


In [ ]:
# Beta binomial Model

def bbm_smoothing(df, trials, success, fake_trials):
    """
    This is a beta-binomial smoothing model. This is a fancier way of finding the rate of something.
    In a situation where we have a small sample size, such as a school district with 10 teachers, because 
    there are so few teachers we have a lower probability of having someone with 30+ years (which is very small); 
    and if we have 3 teachers with 30+ years at another small district, that also is a very small probability, 
    but it could happen. What I want to do is try and help these smaller districts out so they aren't thrown 
    completely off with extreme values in one category when we are grouping down the line.

    df: DataFrame
    Trial: In this case, the labor market of teachers in a given year.
    Success: Teachers with 30+ years of experience.
    fake_trials: How much assistance we give schools when calculating the smoothed rate.
    """
    n_trials = df[trials].sum()
    k_successes = df[success].sum()
    probability = k_successes/n_trials
    fake_successes = probability*fake_trials
    fake_failures = (1-probability)*fake_trials

    df[success+"bbm_smoothed"] = (df[success]+fake_successes)/(df[success] + fake_successes + fake_failures)
    return df


In [8]:
# Function: Calculating z score
def zscore(df: pd.DataFrame):
    """
    Calculates the z score of a column grouped by year
    df: DataFrame
    """
    for col in df.columns:
        if "_Smoothed_Rate" in col:
            z_col = col.replace("_Smoothed_Rate", "") + "_ZScore"
            df[z_col] = (
                df[col] -
                df.groupby('Year')[col].transform('mean')
            ) / df.groupby('Year')[col].transform('std')

    return df

In [9]:
student_df_2 = find_rate(student_df_1, student_numerators, 'All Students Count')
student_df_2 = rate_smoothing(student_df_2, student_numerators, 'All Students Count')

student_df_2 = zscore(student_df_2)


C:\Users\crsal\AppData\Local\Temp\ipykernel_9220\2475992474.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[z_col] = (


In [10]:
type(student_df_2)

pandas.DataFrame

In [11]:
student_df_2

,YearDistrict ID,DISTRICT,Year,All Students Count,Special Ed Count,Bilingual/ESL Count,Career & Technical Education Count,Gifted & Talented Count,EB/EL Count,Econ Disadv Count,...,Female Count_ZScore,Section 504 Count_ZScore,Title I Count_ZScore,Homeless Count_ZScore,Immigrant Count_ZScore,Migrant Count_ZScore,Total Students with Disabilities Count_ZScore,Intellectual Disabilities Count_ZScore,Physical Disabilities Count_ZScore,Behavioral Disabilities Count_ZScore
0,2017001902,001902,2017,576,82,3,150.0,55,3,192,...,NaN,NaN,NaN,NaN,NaN,NaN,1.844961,3.057673,0.474996,-0.422191
1,2017001903,001903,2017,1267,139,19,299.0,39,19,673,...,NaN,NaN,NaN,NaN,NaN,NaN,0.676961,0.778740,0.650738,-0.256123
2,2017001904,001904,2017,846,75,18,229.0,59,18,446,...,NaN,NaN,NaN,NaN,NaN,NaN,-0.100268,0.487562,-1.206285,-0.303734
3,2017001906,001906,2017,377,37,6,118.0,24,6,172,...,NaN,NaN,NaN,NaN,NaN,NaN,0.236574,1.895178,NaN,NaN
4,2017001907,001907,2017,3453,303,516,1222.0,93,557,2587,...,NaN,NaN,NaN,NaN,NaN,NaN,-0.133288,-0.336828,-0.522643,-0.041757
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10327,2025252902,252902,2025,225,42,4,73.0,3,4,140,...,-0.242744,0.412917,0.630018,-0.564989,-0.470923,-0.295356,0.530751,0.340480,-0.297027,0.489243
10328,2025252903,252903,2025,689,145,98,202.0,27,98,383,...,-0.677172,0.761179,-0.232649,0.408007,1.612112,-0.304695,1.114734,-0.377171,3.057363,NaN
10329,2025253901,253901,2025,3282,566,1072,940.0,233,981,2917,...,-0.288713,-0.132358,0.719639,2.012773,-0.202591,0.556153,0.259267,0.511576,-0.449446,-0.710895
10330,2025254901,254901,2025,1635,298,40,575.0,127,41,1329,...,-0.093777,0.314461,0.712491,1.424992,-0.535071,0.210064,0.483649,0.146728,0.018435,0.073879


In [12]:
staff_df_1.columns

Index(['YearDistrict ID', 'Year', 'DISTRICT',
       'Teacher Total Full Time Equiv Count',
       'Support Total Full Time Equiv Count',
       'School Admin Total Full Time Equiv Count',
       'Central Admin Total Full Time Equiv Count',
       'Educ Aide Total Full Time Equiv Count',
       'Teacher Total Base Salary Total', 'Support Total Base Salary Total',
       'School Admin Total Base Salary Total',
       'Teacher Beginning Full Time Equiv Count',
       'Teacher 1-5 Years Full Time Equiv Count',
       'Teacher 6-10 Years Full Time Equiv Count',
       'Teacher 11-20 Years Full Time Equiv Count',
       'Teacher 21-30 Years Full Time Equiv Count',
       'Teacher > 30 Years Full Time Equiv Count',
       'Teacher Beginning Base Salary Total',
       'Teacher 1-5 Years Base Salary Total',
       'Teacher 6-10 Years Base Salary Total',
       'Teacher 11-20 Years Base Salary Total',
       'Teacher 21-30 Years Base Salary Total',
       'Teacher > 30 Years Base Salary Total',

In [13]:
teacher_ep_numerator  =['Teacher Beginning Full Time Equiv Count',
                        'Teacher 1-5 Years Full Time Equiv Count',
                        'Teacher 6-10 Years Full Time Equiv Count',
                        'Teacher 11-20 Years Full Time Equiv Count',
                        'Teacher 21-30 Years Full Time Equiv Count',
                        'Teacher > 30 Years Full Time Equiv Count']
teacher_pay_numerator = ['Teacher Beginning Base Salary Total',
                        'Teacher 1-5 Years Base Salary Total',
                        'Teacher 6-10 Years Base Salary Total',
                        'Teacher 11-20 Years Base Salary Total',
                        'Teacher 21-30 Years Base Salary Total',]

teacher_edu_numerator = ['Teacher No Degree Full Time Equiv Count',
                        'Teacher BA Degree Full Time Equiv Count',
                        'Teacher MS Degree Full Time Equiv Count',
                        'Teacher PH Degree Full Time Equiv Count',]

teacher_eth_numerator = ['Teacher American Indian Full Time Equiv Count',
                        'Teacher Pacific Islander Full Time Equiv Count',
                        'Teacher Asian Full Time Equiv Count',
                        'Teacher African American Full Time Equiv Count',
                        'Teacher Hispanic Full Time Equiv Count',
                        'Teacher White Full Time Equiv Count',
                        'Teacher Two or more races Full Time Equiv Count',]



In [14]:
staff_df_2 = find_rate(staff_df_1, teacher_ep_numerator, 'Teacher Total Full Time Equiv Count')
staff_df_2 = find_rate(staff_df_2, teacher_edu_numerator,'Teacher Total Full Time Equiv Count')
staff_df_2 = find_rate(staff_df_2, teacher_eth_numerator,'Teacher Total Full Time Equiv Count')
staff_df_2 = find_rate(staff_df_2, 'Teacher Total Base Salary Total', 'Teacher Total Full Time Equiv Count')
staff_df_2 = find_rate(staff_df_2, teacher_pay_numerator, 'Teacher Total Base Salary Total')

staff_df_2 = zscore(staff_df_2)

In [15]:
staff_df_2

,YearDistrict ID,Year,DISTRICT,Teacher Total Full Time Equiv Count,Support Total Full Time Equiv Count,School Admin Total Full Time Equiv Count,Central Admin Total Full Time Equiv Count,Educ Aide Total Full Time Equiv Count,Teacher Total Base Salary Total,Support Total Base Salary Total,...,Teacher African American Full Time Equiv Count_Rate,Teacher Hispanic Full Time Equiv Count_Rate,Teacher White Full Time Equiv Count_Rate,Teacher Two or more races Full Time Equiv Count_Rate,Teacher Total Base Salary Total_Rate,Teacher Beginning Base Salary Total_Rate,Teacher 1-5 Years Base Salary Total_Rate,Teacher 6-10 Years Base Salary Total_Rate,Teacher 11-20 Years Base Salary Total_Rate,Teacher 21-30 Years Base Salary Total_Rate
0,2017001902,2017,001902,52.4,6.9,4.1,2.5,13.5,2418873.4,350003.3,...,0.038168,0.019084,0.942748,0.000000,46161.706107,0.016981,0.085689,0.109859,0.381470,NaN
1,2017001903,2017,001903,103.0,7.0,6.0,2.0,26.8,4628074.1,346087.0,...,0.019417,0.048544,0.932039,0.000000,44932.758252,0.013544,0.123166,0.130832,0.479028,NaN
2,2017001904,2017,001904,59.0,5.0,4.0,3.0,19.9,2580639.3,276976.0,...,0.067797,0.016949,0.898305,0.016949,43739.649153,0.023506,0.187102,0.160851,0.301863,NaN
3,2017001906,2017,001906,35.3,2.0,3.0,5.0,7.0,1548155.4,90675.0,...,0.056657,0.000000,0.943343,0.000000,43857.093484,0.007493,0.226358,0.131333,0.418773,NaN
4,2017001907,2017,001907,263.5,28.0,17.0,8.0,74.2,11827846.6,1459280.2,...,0.094118,0.096774,0.786338,0.015180,44887.463378,0.063850,0.256080,0.194728,0.226025,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10327,2025252902,2025,252902,24.8,0.7,2.0,1.0,10.2,1479470.0,49629.0,...,0.000000,0.080645,0.919355,0.000000,59656.048387,0.050234,0.103580,0.232203,0.216273,0.379130
10328,2025252903,2025,252903,73.3,7.7,6.3,1.0,1.0,4098574.3,515342.9,...,0.027285,0.079127,0.881310,0.000000,55915.065484,0.037278,0.235070,0.173941,0.264990,0.153651
10329,2025253901,2025,253901,240.2,47.4,14.3,11.0,62.7,13664248.8,3345159.2,...,0.000000,0.970858,0.024979,0.004163,56886.964197,0.129147,0.216710,0.105054,0.334085,0.180021
10330,2025254901,2025,254901,107.8,22.9,13.2,7.0,62.8,6129788.5,1547240.3,...,0.009276,0.925788,0.064935,0.000000,56862.602041,0.112349,0.239695,0.152630,0.285322,0.175241


In [16]:
# Select Teacher Numerators

